# 🌿 CSIRO Biomass Training - Improved Version

## 🚀 改良点
1. **SWA (Stochastic Weight Averaging)** - エポック25から開始
2. **マルチスケール学習** - [448, 512, 576]のサイクル
3. **改良された可視化** - 学習履歴、画像サイズ、学習率の推移
4. **EMAとSWAの両モデル保存** - より良い汎化性能

## ⏰ 実行手順
1. セル1: GPU確認 & パッケージインストール
2. セル2-4: Kaggle API設定 & データダウンロード
3. セル5: 全ての定義（Import, Model, Dataset, Training）
4. セル6-7: データ読み込み & 学習開始

## 1. 🔧 Environment Setup

In [ ]:
#@title 1.1 Check GPU & Install Packages

# GPU確認
!nvidia-smi

# パッケージインストール（改良版で必要なもの全て）
!pip install -q timm>=1.0.0 albumentations kaggle pandas scikit-learn opencv-python-headless matplotlib tqdm unzip

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu} ({vram:.1f} GB)")

import timm
print(f"timm: {timm.__version__}")

# モデル確認
model_name = "vit_huge_plus_patch16_dinov3.lvd1689m"
available = model_name in timm.list_pretrained()
print(f"{model_name}: {'✅' if available else '❌'}")

# 改良版の新機能確認
try:
    from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
    print("✅ SWA support available")
except ImportError:
    print("❌ SWA not available - need PyTorch >= 1.6")

# 他のパッケージ確認
import pandas as pd
import sklearn
import cv2
import albumentations
print(f"pandas: {pd.__version__} ✅")
print(f"sklearn: {sklearn.__version__} ✅")
print(f"cv2: {cv2.__version__} ✅")
print(f"albumentations: {albumentations.__version__} ✅")

## 2. 📁 Kaggle API & Dataset

### kaggle.json の設定方法
1. https://www.kaggle.com/settings → API → Create New API Token
2. ダウンロードした `kaggle.json` を JupyterLab にアップロード
3. セル2.2 を実行

In [ ]:
#@title 2.1 Check Kaggle API Status
import os

kaggle_dir = os.path.expanduser("~/.kaggle")
kaggle_json = os.path.join(kaggle_dir, "kaggle.json")

if os.path.exists(kaggle_json):
    print("✅ Kaggle API already configured")
else:
    print("❌ kaggle.json が見つかりません")
    print("")
    print("📤 kaggle.json をアップロードしてください:")
    print("   1. JupyterLabの左パネルからファイルをアップロード")
    print("   2. 次のセル（2.2）を実行")

In [ ]:
#@title 2.2 Configure Kaggle API（kaggle.jsonアップロード後に実行）
import os
import shutil

kaggle_dir = os.path.expanduser("~/.kaggle")
os.makedirs(kaggle_dir, exist_ok=True)

# ファイルを探して移動
kaggle_json_dest = os.path.join(kaggle_dir, "kaggle.json")
found = False

# 検索する場所のリスト
search_paths = [
    "/workspace/kaggle.json",
    "kaggle.json",
    os.path.expanduser("~/kaggle.json"),
]

for path in search_paths:
    if os.path.exists(path):
        shutil.copy(path, kaggle_json_dest)
        os.chmod(kaggle_json_dest, 0o600)
        print(f"✅ Found and configured: {path}")
        found = True
        break

if not found and os.path.exists(kaggle_json_dest):
    print("✅ Kaggle API already configured")
    found = True

if found:
    # APIテスト
    import subprocess
    result = subprocess.run(["kaggle", "competitions", "list", "-s", "csiro"], 
                           capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ API動作確認OK")
    else:
        print("❌ APIエラー")
        print(f"   {result.stderr}")
else:
    print("❌ kaggle.json が見つかりません")
    print("   JupyterLabにkaggle.jsonをアップロードしてから再実行してください")

In [ ]:
#@title 2.3 Download Dataset
import os

DATA_DIR = "/workspace/csiro-biomass"
TRAIN_DIR = os.path.join(DATA_DIR, "train")

if os.path.exists(TRAIN_DIR) and len(os.listdir(TRAIN_DIR)) > 0:
    print(f"✅ Dataset already exists: {len(os.listdir(TRAIN_DIR))} images")
else:
    print("📥 Downloading dataset...")
    os.makedirs(DATA_DIR, exist_ok=True)
    
    !kaggle competitions download -c csiro-biomass -p /workspace/
    
    if os.path.exists("/workspace/csiro-biomass.zip"):
        print("📦 Extracting...")
        !unzip -q /workspace/csiro-biomass.zip -d {DATA_DIR}
        !rm /workspace/csiro-biomass.zip
        
        if os.path.exists(TRAIN_DIR):
            print(f"✅ Dataset ready: {len(os.listdir(TRAIN_DIR))} train images")
        else:
            print("❌ Extraction failed")
            !ls -la {DATA_DIR}
    else:
        print("❌ Download failed!")
        print("")
        print("⚠️ 以下を確認してください:")
        print("   1. kaggle.json が正しく設定されているか")
        print("   2. コンペのルールに同意しているか:")
        print("      https://www.kaggle.com/competitions/csiro-biomass/rules")

## 3. ⚙️ Improved Training Setup

改良版の全ての定義（SWA、マルチスケール、改良されたモデルなど）をこのセルで行います。

In [ ]:
#@title 3.1 All Imports, Configuration, Model, Dataset, and Training Functions (Improved Version)

# ============================================================
# IMPORTS
# ============================================================
import os
import gc
import math
import random
import time
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Optional, Tuple

import cv2
from tqdm.auto import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import LambdaLR
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm
from timm.utils import ModelEmaV2
from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")


# ============================================================
# IMPROVED CONFIGURATION
# ============================================================
class CFG:
    TARGETS = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
    N_FOLDS = 5
    BACKBONE = "vit_huge_plus_patch16_dinov3.lvd1689m"
    
    # Multi-scale settings (NEW!)
    IMG_SIZES = [448, 512, 576]  # 512±64
    BASE_IMG_SIZE = 512
    
    # Paths
    DATA_DIR = Path("/workspace/csiro-biomass")
    TRAIN_CSV = DATA_DIR / "train.csv"
    TRAIN_IMAGE_DIR = DATA_DIR / "train"
    CHECKPOINT_DIR = Path("/workspace/checkpoints_improved")  # 改良版用ディレクトリ
    
    SEED = 42
    BATCH_SIZE = 2
    GRAD_ACC = 4
    NUM_WORKERS = 4
    EPOCHS = 30
    WARMUP_EPOCHS = 3
    PATIENCE = 5
    
    # SWA settings (NEW!)
    SWA_START_EPOCH = 25  # Start SWA at epoch 25
    SWA_LR = 1e-5  # SWA learning rate
    
    LR_BACKBONE = 5e-5
    LR_HEAD = 1e-4
    WD = 1e-2
    EMA_DECAY = 0.995
    
    LOSS_WEIGHTS = np.array([0.2, 0.2, 0.2, 0.2, 0.2])
    R2_WEIGHTS = np.array([0.1, 0.1, 0.1, 0.2, 0.5])
    
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


CFG.CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=CFG.SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True


seed_everything()

print("=" * 60)
print("📋 Improved Training Configuration")
print("=" * 60)
print(f"N_FOLDS       = {CFG.N_FOLDS}")
print(f"BACKBONE      = {CFG.BACKBONE}")
print(f"IMG_SIZES     = {CFG.IMG_SIZES} (Multi-scale!)")
print(f"BATCH_SIZE    = {CFG.BATCH_SIZE} x {CFG.GRAD_ACC} = {CFG.BATCH_SIZE * CFG.GRAD_ACC}")
print(f"SWA_START     = Epoch {CFG.SWA_START_EPOCH}")
print(f"DEVICE        = {CFG.DEVICE}")
print("-" * 60)
csv_ok = "✅" if CFG.TRAIN_CSV.exists() else "❌"
dir_ok = "✅" if CFG.TRAIN_IMAGE_DIR.exists() else "❌"
print(f"TRAIN_CSV     = {CFG.TRAIN_CSV} {csv_ok}")
print(f"TRAIN_DIR     = {CFG.TRAIN_IMAGE_DIR} {dir_ok}")
print(f"CKPT_DIR      = {CFG.CHECKPOINT_DIR} ✅")
print("=" * 60)

# ============================================================
# MODEL (Same as before)
# ============================================================
class LocalMambaBlock(nn.Module):
    def __init__(self, dim, kernel_size=5, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.dwconv = nn.Conv1d(dim, dim, kernel_size=kernel_size, padding=kernel_size//2, groups=dim)
        self.gate = nn.Linear(dim, dim)
        self.proj = nn.Linear(dim, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        shortcut = x
        x = self.norm(x)
        x = x * torch.sigmoid(self.gate(x))
        x = self.dwconv(x.transpose(1, 2)).transpose(1, 2)
        x = self.proj(x)
        return shortcut + self.drop(x)


class BiomassModel(nn.Module):
    def __init__(self, model_name, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0, global_pool="")
        nf = self.backbone.num_features
        
        if hasattr(self.backbone, "set_grad_checkpointing"):
            self.backbone.set_grad_checkpointing(True)
            print("✅ Gradient Checkpointing enabled")
        
        self.fusion = nn.Sequential(
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1),
            LocalMambaBlock(nf, kernel_size=5, dropout=0.1)
        )
        self.pool = nn.AdaptiveAvgPool1d(1)
        
        self.head_green = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_dead = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )
        self.head_clover = nn.Sequential(
            nn.Linear(nf, nf//2), nn.GELU(), nn.Dropout(0.2), 
            nn.Linear(nf//2, 1), nn.Softplus()
        )

    def forward(self, x):
        left, right = x
        x_l = self.backbone(left)
        x_r = self.backbone(right)
        x = self.fusion(torch.cat([x_l, x_r], dim=1))
        x = self.pool(x.transpose(1, 2)).flatten(1)
        green = self.head_green(x)
        dead = self.head_dead(x)
        clover = self.head_clover(x)
        gdm = green + clover
        total = green + clover + dead
        return torch.cat([green, dead, clover, gdm, total], dim=1)


print("✅ Model defined")

# ============================================================
# DATASET & TRANSFORMS (Multi-scale support)
# ============================================================
class BiomassDataset(Dataset):
    def __init__(self, df, transforms, image_dir):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms
        self.image_dir = Path(image_dir)

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self.image_dir / Path(row["image_path"]).name
        img = cv2.imread(str(img_path))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        left = img[:, :w//2]
        right = img[:, w//2:]
        
        if self.transforms:
            seed = random.randint(0, 99999)
            random.seed(seed)
            np.random.seed(seed)
            left = self.transforms(image=left)["image"]
            random.seed(seed)
            np.random.seed(seed)
            right = self.transforms(image=right)["image"]
        
        labels = torch.tensor([row[t] for t in CFG.TARGETS], dtype=torch.float32)
        return left, right, labels


def get_train_transforms(img_size=None):
    """Get training transforms with dynamic image size"""
    if img_size is None:
        img_size = CFG.BASE_IMG_SIZE
    
    return A.Compose([
        A.Resize(img_size, img_size),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=15, p=0.5),
        A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05, p=0.5),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


def get_val_transforms(img_size=None):
    """Get validation transforms with dynamic image size"""
    if img_size is None:
        img_size = CFG.BASE_IMG_SIZE
        
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])


print("✅ Dataset & Transforms defined (Multi-scale support)")

# ============================================================
# LOSS & METRICS (Same as before)
# ============================================================
def biomass_loss(outputs, labels, weights=None):
    losses = nn.SmoothL1Loss(reduction="none", beta=5.0)(outputs, labels).mean(dim=0)
    if weights is None:
        return losses.mean()
    w = torch.as_tensor(weights, device=losses.device, dtype=losses.dtype)
    return (losses * w / w.sum()).sum()


def weighted_r2_score(y_true, y_pred):
    r2s = []
    for i in range(y_true.shape[1]):
        ss_res = np.sum((y_true[:, i] - y_pred[:, i]) ** 2)
        ss_tot = np.sum((y_true[:, i] - np.mean(y_true[:, i])) ** 2)
        r2s.append(1 - ss_res / ss_tot if ss_tot > 0 else 0.0)
    r2s = np.array(r2s)
    return np.sum(r2s * CFG.R2_WEIGHTS) / np.sum(CFG.R2_WEIGHTS), r2s


print("✅ Loss & Metrics defined")

# ============================================================
# TRAINING FUNCTIONS (Improved with SWA support)
# ============================================================
scaler = torch.cuda.amp.GradScaler()


def train_epoch(model, loader, optimizer, device, epoch, ema=None, swa_model=None):
    """Training epoch with multi-scale and SWA support"""
    model.train()
    total_loss = 0
    optimizer.zero_grad()
    
    # Select image size for this epoch (cycle through sizes)
    img_size = CFG.IMG_SIZES[epoch % len(CFG.IMG_SIZES)]
    pbar = tqdm(loader, desc=f"Train [Size: {img_size}]", leave=False)
    
    for i, (left, right, labels) in enumerate(pbar):
        left = left.to(device)
        right = right.to(device)
        labels = labels.to(device)
        
        with torch.cuda.amp.autocast():
            outputs = model((left, right))
            loss = biomass_loss(outputs, labels, CFG.LOSS_WEIGHTS) / CFG.GRAD_ACC
        
        scaler.scale(loss).backward()
        total_loss += loss.item() * left.size(0) * CFG.GRAD_ACC
        
        if (i + 1) % CFG.GRAD_ACC == 0 or (i + 1) == len(loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            
            if ema:
                ema.update(model)
                
            # Update SWA model if in SWA phase
            if swa_model is not None and epoch >= CFG.SWA_START_EPOCH:
                swa_model.update_parameters(model)
                
            optimizer.zero_grad()
        
        pbar.set_postfix({"loss": f"{loss.item() * CFG.GRAD_ACC:.4f}"})
    
    return total_loss / len(loader.dataset)


@torch.no_grad()
def valid_epoch(model, loader, device):
    """Validation epoch"""
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    
    for left, right, labels in tqdm(loader, desc="Valid", leave=False):
        left = left.to(device)
        right = right.to(device)
        labels = labels.to(device)
        
        with torch.cuda.amp.autocast():
            outputs = model((left, right))
            loss = biomass_loss(outputs, labels, CFG.LOSS_WEIGHTS)
        
        total_loss += loss.item() * left.size(0)
        all_preds.append(outputs.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
    
    preds = np.vstack(all_preds)
    labels = np.vstack(all_labels)
    r2, per_r2 = weighted_r2_score(labels, preds)
    return total_loss / len(loader.dataset), r2, per_r2


def build_optimizer(model):
    backbone_ids = {id(p) for p in model.backbone.parameters()}
    backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) in backbone_ids]
    head_params = [p for p in model.parameters() if p.requires_grad and id(p) not in backbone_ids]
    return optim.AdamW([
        {"params": backbone_params, "lr": CFG.LR_BACKBONE, "weight_decay": CFG.WD},
        {"params": head_params, "lr": CFG.LR_HEAD, "weight_decay": CFG.WD},
    ])


def build_scheduler(optimizer, num_epochs):
    def lr_lambda(epoch):
        if epoch < CFG.WARMUP_EPOCHS:
            return (epoch + 1) / CFG.WARMUP_EPOCHS
        progress = (epoch - CFG.WARMUP_EPOCHS) / (num_epochs - CFG.WARMUP_EPOCHS)
        return 0.5 * (1 + math.cos(math.pi * progress))
    return LambdaLR(optimizer, lr_lambda)


print("✅ Training functions defined (SWA support added)")
print("\n🎉 All improved definitions complete!")

## 4. 📊 Load Data & Setup

In [ ]:
#@title 4.1 Load Data
print("📊 Loading data...")

if not CFG.TRAIN_CSV.exists():
    print("❌ train.csv が見つかりません！")
    print("   セクション 2.3 を再実行してください")
else:
    df_long = pd.read_csv(CFG.TRAIN_CSV)
    df_wide = df_long.pivot(index="image_path", columns="target_name", values="target").reset_index()
    df_wide = df_wide[["image_path"] + CFG.TARGETS]
    meta = df_long[["image_path", "Sampling_Date", "State"]].drop_duplicates()
    df_wide = df_wide.merge(meta, on="image_path", how="left")
    
    print(f"✅ Loaded {len(df_wide)} images")
    print(f"   Columns: {list(df_wide.columns)}")
    
    # Check existing checkpoints
    print("\n📁 Existing improved checkpoints:")
    for fold in range(CFG.N_FOLDS):
        ema_ckpt = CFG.CHECKPOINT_DIR / f"best_ema_fold{fold}.pth"
        swa_ckpt = CFG.CHECKPOINT_DIR / f"best_swa_fold{fold}.pth"
        
        ema_status = "✅" if ema_ckpt.exists() else "❌"
        swa_status = "✅" if swa_ckpt.exists() else "❌"
        
        if ema_ckpt.exists():
            ema_size = ema_ckpt.stat().st_size / 1024**3
            print(f"  Fold {fold} EMA: {ema_status} ({ema_size:.2f} GB)")
        else:
            print(f"  Fold {fold} EMA: {ema_status}")
            
        if swa_ckpt.exists():
            swa_size = swa_ckpt.stat().st_size / 1024**3
            print(f"  Fold {fold} SWA: {swa_status} ({swa_size:.2f} GB)")
        else:
            print(f"  Fold {fold} SWA: {swa_status}")

## 5. 🚀 Improved Training Function with SWA

In [ ]:
#@title 5.1 Train Fold Function (Improved with SWA)

def train_fold_improved(fold, df_wide):
    print(f"\n{'='*60}")
    print(f"🚀 FOLD {fold} / {CFG.N_FOLDS-1} - Improved Version")
    print(f"   Features: SWA, Multi-scale Training")
    print(f"{'='*60}")
    
    # K-Fold split
    sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
    splits = list(sgkf.split(df_wide, df_wide["State"], groups=df_wide["Sampling_Date"]))
    train_idx, val_idx = splits[fold]
    
    train_df = df_wide.iloc[train_idx].reset_index(drop=True)
    val_df = df_wide.iloc[val_idx].reset_index(drop=True)
    print(f"Train: {len(train_df)} | Val: {len(val_df)}")
    
    # Clear memory
    torch.cuda.empty_cache()
    gc.collect()
    
    # Model
    model = BiomassModel(CFG.BACKBONE, pretrained=True).to(CFG.DEVICE)
    ema = ModelEmaV2(model, decay=CFG.EMA_DECAY)
    optimizer = build_optimizer(model)
    scheduler = build_scheduler(optimizer, CFG.EPOCHS)
    
    # SWA model (initialized later)
    swa_model = None
    swa_scheduler = None
    swa_start = CFG.SWA_START_EPOCH
    
    # Training loop
    best_r2 = -float("inf")
    best_swa_r2 = -float("inf")
    patience_counter = 0
    history = {"train_loss": [], "val_loss": [], "val_r2": [], "img_size": []}
    
    for epoch in range(CFG.EPOCHS):
        # Dynamic image size selection
        img_size = CFG.IMG_SIZES[epoch % len(CFG.IMG_SIZES)]
        
        # Create datasets with current image size
        train_ds = BiomassDataset(
            train_df, 
            get_train_transforms(img_size), 
            CFG.TRAIN_IMAGE_DIR
        )
        val_ds = BiomassDataset(
            val_df, 
            get_val_transforms(CFG.BASE_IMG_SIZE),  # Always validate at base size
            CFG.TRAIN_IMAGE_DIR
        )
        
        train_loader = DataLoader(
            train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True,
            num_workers=CFG.NUM_WORKERS, pin_memory=True, drop_last=True
        )
        val_loader = DataLoader(
            val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False,
            num_workers=CFG.NUM_WORKERS, pin_memory=True
        )
        
        print(f"\nEpoch {epoch+1}/{CFG.EPOCHS} [Image Size: {img_size}]")
        
        # Initialize SWA if we've reached the SWA start epoch
        if epoch == swa_start and swa_model is None:
            print(f"  🔄 Starting SWA at epoch {epoch+1}")
            swa_model = AveragedModel(model)
            swa_scheduler = SWALR(optimizer, swa_lr=CFG.SWA_LR)
        
        # Train
        train_loss = train_epoch(model, train_loader, optimizer, CFG.DEVICE, epoch, ema, swa_model)
        
        # Step scheduler
        if epoch < swa_start:
            scheduler.step()
        elif swa_scheduler is not None:
            swa_scheduler.step()
        
        # Validate with EMA model
        val_loss, val_r2, per_r2 = valid_epoch(ema.module, val_loader, CFG.DEVICE)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_r2"].append(val_r2)
        history["img_size"].append(img_size)
        
        lr = optimizer.param_groups[0]["lr"]
        print(f"  Loss: {train_loss:.4f}/{val_loss:.4f} | R2: {val_r2:.4f} | LR: {lr:.2e}")
        print(f"  R2: G={per_r2[0]:.3f} D={per_r2[1]:.3f} C={per_r2[2]:.3f} GDM={per_r2[3]:.3f} T={per_r2[4]:.3f}")
        
        # Save best EMA model
        if val_r2 > best_r2:
            best_r2 = val_r2
            patience_counter = 0
            save_path = CFG.CHECKPOINT_DIR / f"best_ema_fold{fold}.pth"
            torch.save(ema.module.state_dict(), save_path)
            print(f"  ✅ Saved EMA: {save_path}")
        else:
            patience_counter += 1
            if patience_counter >= CFG.PATIENCE and epoch < swa_start:
                print(f"\n⚠️ Early stopping at epoch {epoch+1}")
                break
        
        # Validate SWA model if available
        if swa_model is not None and epoch >= swa_start:
            # Update batch norm statistics
            update_bn(train_loader, swa_model, device=CFG.DEVICE)
            swa_loss, swa_r2, swa_per_r2 = valid_epoch(swa_model, val_loader, CFG.DEVICE)
            print(f"  SWA R2: {swa_r2:.4f}")
            
            if swa_r2 > best_swa_r2:
                best_swa_r2 = swa_r2
                save_path = CFG.CHECKPOINT_DIR / f"best_swa_fold{fold}.pth"
                torch.save(swa_model.state_dict(), save_path)
                print(f"  ✅ Saved SWA: {save_path}")
    
    print(f"\n🏆 Fold {fold} Results:")
    print(f"   Best EMA R2: {best_r2:.4f}")
    if best_swa_r2 > -float("inf"):
        print(f"   Best SWA R2: {best_swa_r2:.4f}")
    
    # Plot training history
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Loss plot
    axes[0, 0].plot(history["train_loss"], label="Train")
    axes[0, 0].plot(history["val_loss"], label="Val")
    axes[0, 0].set_title(f"Fold {fold} Loss")
    axes[0, 0].set_xlabel("Epoch")
    axes[0, 0].set_ylabel("Loss")
    axes[0, 0].legend()
    axes[0, 0].grid(True)
    
    # R2 plot
    axes[0, 1].plot(history["val_r2"], "g")
    axes[0, 1].axhline(best_r2, color="r", linestyle="--", label=f"Best EMA: {best_r2:.4f}")
    if best_swa_r2 > -float("inf"):
        axes[0, 1].axhline(best_swa_r2, color="b", linestyle="--", label=f"Best SWA: {best_swa_r2:.4f}")
    axes[0, 1].axvline(swa_start, color="gray", linestyle=":", label=f"SWA Start")
    axes[0, 1].set_title(f"Fold {fold} R2")
    axes[0, 1].set_xlabel("Epoch")
    axes[0, 1].set_ylabel("R2 Score")
    axes[0, 1].legend()
    axes[0, 1].grid(True)
    
    # Image size plot
    axes[1, 0].plot(history["img_size"], "o-")
    axes[1, 0].set_title("Multi-scale Schedule")
    axes[1, 0].set_xlabel("Epoch")
    axes[1, 0].set_ylabel("Image Size")
    axes[1, 0].grid(True)
    
    # Learning rate plot (approximation)
    lrs = []
    for e in range(len(history["train_loss"])):
        if e < CFG.WARMUP_EPOCHS:
            lr_factor = (e + 1) / CFG.WARMUP_EPOCHS
        else:
            progress = (e - CFG.WARMUP_EPOCHS) / (CFG.EPOCHS - CFG.WARMUP_EPOCHS)
            lr_factor = 0.5 * (1 + math.cos(math.pi * progress))
        lrs.append(CFG.LR_BACKBONE * lr_factor)
    
    axes[1, 1].plot(lrs)
    axes[1, 1].set_title("Learning Rate Schedule")
    axes[1, 1].set_xlabel("Epoch")
    axes[1, 1].set_ylabel("Learning Rate")
    axes[1, 1].set_yscale("log")
    axes[1, 1].grid(True)
    
    plt.tight_layout()
    plt.savefig(CFG.CHECKPOINT_DIR / f"fold{fold}_history.png", dpi=100)
    plt.show()
    
    # Cleanup
    del model, ema, swa_model, optimizer, scheduler, train_loader, val_loader
    torch.cuda.empty_cache()
    gc.collect()
    
    return best_r2, best_swa_r2

print("✅ Improved train_fold function defined")

## 6. 🏃‍♂️ Execute Training

In [ ]:
#@title 6.1 🚀 Train All Folds (Improved Version)
print("=" * 60)
print("🚀 STARTING IMPROVED TRAINING - ALL 5 FOLDS")
print("=" * 60)
print(f"\n⏰ Estimated time: ~18-22 hours (RTX A5000)")
print(f"💰 Estimated cost: ~$14-17 (RTX A5000 on RunPod)")
print(f"🔥 Features: SWA, Multi-scale, Enhanced EMA")
print("\n" + "=" * 60)

all_scores = []
start_time = time.time()

for fold in range(CFG.N_FOLDS):
    # Skip if both EMA and SWA already trained
    ema_ckpt = CFG.CHECKPOINT_DIR / f"best_ema_fold{fold}.pth"
    swa_ckpt = CFG.CHECKPOINT_DIR / f"best_swa_fold{fold}.pth"
    
    if ema_ckpt.exists() and swa_ckpt.exists():
        ema_size = ema_ckpt.stat().st_size / 1024**3
        swa_size = swa_ckpt.stat().st_size / 1024**3
        print(f"\n⏭️ Fold {fold} already complete:")
        print(f"   EMA: {ema_size:.2f} GB | SWA: {swa_size:.2f} GB")
        continue
    
    fold_start = time.time()
    ema_score, swa_score = train_fold_improved(fold, df_wide)
    fold_time = (time.time() - fold_start) / 3600
    
    all_scores.append((fold, ema_score, swa_score, fold_time))
    print(f"\n⏱️ Fold {fold} completed in {fold_time:.2f} hours")

total_time = (time.time() - start_time) / 3600

print("\n" + "=" * 60)
print("🎉 IMPROVED TRAINING COMPLETE!")
print("=" * 60)
for fold, ema_score, swa_score, t in all_scores:
    print(f"  Fold {fold}: EMA R2={ema_score:.4f}, SWA R2={swa_score:.4f} ({t:.2f}h)")
print(f"\n⏱️ Total time: {total_time:.2f} hours")
print(f"📁 Checkpoints saved to: {CFG.CHECKPOINT_DIR}")

# Calculate average scores
if all_scores:
    avg_ema = np.mean([score[1] for score in all_scores])
    avg_swa = np.mean([score[2] for score in all_scores if score[2] > -float('inf')])
    print(f"\n📊 Average Scores:")
    print(f"   EMA: {avg_ema:.4f}")
    if avg_swa > -float('inf'):
        print(f"   SWA: {avg_swa:.4f}")
        print(f"   Improvement: {((avg_swa - avg_ema) / avg_ema * 100):+.2f}%")

## 7. 📥 Verify & Download Models

学習完了後、チェックポイントを確認してKaggleにアップロードします。

In [ ]:
#@title 7.1 Verify Improved Checkpoints
print("📁 Improved model checkpoints:")
print("-" * 60)

ema_total_size = 0
swa_total_size = 0

for fold in range(CFG.N_FOLDS):
    ema_ckpt = CFG.CHECKPOINT_DIR / f"best_ema_fold{fold}.pth"
    swa_ckpt = CFG.CHECKPOINT_DIR / f"best_swa_fold{fold}.pth"
    
    print(f"Fold {fold}:")
    
    if ema_ckpt.exists():
        ema_size = ema_ckpt.stat().st_size / 1024**3
        ema_total_size += ema_size
        print(f"  ✅ EMA: best_ema_fold{fold}.pth ({ema_size:.2f} GB)")
    else:
        print(f"  ❌ EMA: NOT FOUND")
        
    if swa_ckpt.exists():
        swa_size = swa_ckpt.stat().st_size / 1024**3
        swa_total_size += swa_size
        print(f"  ✅ SWA: best_swa_fold{fold}.pth ({swa_size:.2f} GB)")
    else:
        print(f"  ❌ SWA: NOT FOUND")

print("-" * 60)
print(f"Total EMA models: {ema_total_size:.2f} GB")
print(f"Total SWA models: {swa_total_size:.2f} GB")
print(f"Grand Total: {ema_total_size + swa_total_size:.2f} GB")
print("\n💡 These models are compatible with csiro_inference_improved.ipynb")

In [ ]:
#@title 7.2 Test Load Improved Models
TEST_FOLD = 0

print(f"Testing improved models for fold {TEST_FOLD}...")

# Test EMA model
ema_ckpt = CFG.CHECKPOINT_DIR / f"best_ema_fold{TEST_FOLD}.pth"
if ema_ckpt.exists():
    print(f"\nTesting EMA model...")
    test_model = BiomassModel(CFG.BACKBONE, pretrained=False)
    test_model.load_state_dict(torch.load(ema_ckpt, map_location="cpu"))
    test_model.eval()
    
    with torch.no_grad():
        dummy_left = torch.randn(1, 3, 512, 512)
        dummy_right = torch.randn(1, 3, 512, 512)
        out = test_model((dummy_left, dummy_right))
    
    print(f"✅ EMA Output shape: {out.shape} (expected: [1, 5])")
    del test_model
else:
    print(f"❌ EMA model not found: {ema_ckpt}")

# Test SWA model
swa_ckpt = CFG.CHECKPOINT_DIR / f"best_swa_fold{TEST_FOLD}.pth"
if swa_ckpt.exists():
    print(f"\nTesting SWA model...")
    test_model = BiomassModel(CFG.BACKBONE, pretrained=False)
    test_model.load_state_dict(torch.load(swa_ckpt, map_location="cpu"))
    test_model.eval()
    
    with torch.no_grad():
        dummy_left = torch.randn(1, 3, 512, 512)
        dummy_right = torch.randn(1, 3, 512, 512)
        out = test_model((dummy_left, dummy_right))
    
    print(f"✅ SWA Output shape: {out.shape} (expected: [1, 5])")
    del test_model
else:
    print(f"❌ SWA model not found: {swa_ckpt}")

torch.cuda.empty_cache()
print("\n✅ All improved models load correctly!")

## 🎯 Next Steps

学習が完了したら：

1. **モデルをダウンロード** - JupyterLabで右クリック → Download
2. **Kaggleにアップロード** - DatasetとしてEMAとSWAモデルをアップロード
3. **推論実行** - `csiro_inference_improved.ipynb` を使用
4. **TTA有効** - 1-2%の精度向上のため

### 期待される改善効果
- **SWA**: +1-3%の汎化性能向上
- **マルチスケール**: +1-2%のスケール不変性
- **TTA (推論時)**: +1-2%の精度向上
- **合計**: +3-7%の性能向上

### GPU要件
- **最小**: RTX 3090/4090/A5000 (24GB VRAM)
- **推奨**: RTX A5000 (安定性) または A100 (速度)
- **学習時間**: 18-22時間 (A5000)